In [1]:
# 1. Configuración del entorno
import os

JDBC_DIR = r"C:\PySpark_AdventureWorks\jdbc"
os.environ["PATH"] = JDBC_DIR + os.pathsep + os.environ["PATH"]
print("COMPLETADO")

COMPLETADO


In [2]:
# 2. Inicialización de Spark
from pyspark.sql import SparkSession

JDBC_JAR = r"C:\PySpark_AdventureWorks\jdbc\mssql-jdbc-13.6.0.jre11.jar"

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Spark_AdventureWorks")
    .config("spark.jars", JDBC_JAR)
    .getOrCreate()
)

print("Spark:", spark.version)

Spark: 4.1.3


In [3]:
# 3. Conexión con SQL Server
SERVER = "DESKTOP-LOUTN8G"
DATABASE = "AdventureWorksDW_PC5"

JDBC_URL = (
    f"jdbc:sqlserver://{SERVER};"
    f"databaseName={DATABASE};"
    "integratedSecurity=true;"
    "encrypt=false;"
)

print(JDBC_URL)

jdbc:sqlserver://DESKTOP-LOUTN8G;databaseName=AdventureWorksDW_PC5;integratedSecurity=true;encrypt=false;


In [4]:
# 4. Carga de FactVentas mediante JDBC
ventas_df = (
    spark.read
    .format("jdbc")
    .option("url", JDBC_URL)
    .option("dbtable", "dbo.FactVentas")
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

print("Registros:", ventas_df.count())

Registros: 60398


In [5]:
# 5. Registro como vista temporal para SparkSQL
ventas_df.createOrReplaceTempView("ventas")
print("COMPLETADO")

COMPLETADO


In [6]:
# 6. Agregación de indicadores
spark.sql("SELECT COUNT(*) AS Registros FROM ventas").show()

+---------+
|Registros|
+---------+
|    60398|
+---------+



In [7]:
resultado_kpi = spark.sql("""
    SELECT
        SUM(ImporteVenta) AS VentasTotales,
        SUM(ImporteVenta) - SUM(CostoTotalProducto) AS MargenBruto,
        SUM(Cantidad) AS CantidadVendida
    FROM ventas
""")

resultado_kpi.show()

+-------------+-------------+---------------+
|VentasTotales|  MargenBruto|CantidadVendida|
+-------------+-------------+---------------+
|29358677.2207|12080883.6450|          60398|
+-------------+-------------+---------------+



In [8]:
# 7. Medición del tiempo de ejecución de Spark
import time

inicio = time.perf_counter()

resultado_spark = spark.sql("""
    SELECT
        SUM(ImporteVenta) AS VentasTotales,
        SUM(ImporteVenta) - SUM(CostoTotalProducto) AS MargenBruto,
        SUM(Cantidad) AS CantidadVendida
    FROM ventas
""")

resultado_spark.collect()

fin = time.perf_counter()

tiempo_spark = (fin - inicio) * 1000

print(f"Tiempo de ejecución Spark: {tiempo_spark:.2f} ms")

Tiempo de ejecución Spark: 332.19 ms
